# Embeddings y Búsqueda Vectorial
## Motor de búsqueda semántica de ofertas laborales

**Proyecto Integrador 1 — Ingeniería de Sistemas**

### Objetivo del notebook

A partir del dataset ya limpio (generado en `02_Preprocesamiento.ipynb`), cubrir:

1. **Primera iteración de representación semántica** con Sentence-BERT (Sentence Transformers).
2. **Búsqueda vectorial baseline** con FAISS.
3. **Exploración de bases de datos vectoriales** (Pinecone, Chroma y alternativas) frente a FAISS.

Este notebook asume que ya ejecutaste `01_EDA.ipynb` y `02_Preprocesamiento.ipynb` al menos una vez (para que el parquet limpio exista en Google Drive).

## 0. Preparación e importaciones

Se instalan/importan las librerías necesarias y se monta Google Drive para acceder al dataset limpio guardado en el notebook anterior.

In [ ]:
!pip install -q sentence-transformers faiss-cpu tqdm pinecone-client chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [ ]:
import os
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## 1. Carga del dataset limpio

Se monta Google Drive (misma carpeta `proyecto_integrador` usada en el notebook de preprocesamiento) y se carga el parquet ya limpio, en vez de volver a descargar y limpiar el CSV crudo.

In [ ]:
from pathlib import Path

MONTAR_DRIVE = True  # cambia a False si prefieres trabajar solo con el almacenamiento efimero de Colab

if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/proyecto_integrador")
else:
    BASE_DIR = Path("/content")

RUTA_PROCESSED = BASE_DIR / "data" / "processed"
RUTA_LIMPIO = RUTA_PROCESSED / "job_descriptions_clean.parquet"

if not RUTA_LIMPIO.exists():
    raise FileNotFoundError(
        f"No se encontró {RUTA_LIMPIO}. "
        "Ejecuta primero 02_Preprocesamiento.ipynb para generarlo."
    )

df_clean = pd.read_parquet(RUTA_LIMPIO)
print(f"Registros cargados: {len(df_clean):,}")
df_clean[["Job Title", "texto_combinado"]].head(2)


Mounted at /content/drive
Registros cargados: 1,615,940


,Job Title,texto_combinado
0,Digital Marketing Specialist,"Digital Marketing Specialist Digital Marketing Specialist Social Media Manager Social media platforms (e.g., Faceboo..."
1,Web Developer,"Web Developer Web Developer Frontend Web Developer HTML, CSS, JavaScript Frontend frameworks (e.g., React, Angular) ..."


## 2. Primera iteración de embeddings con Sentence-BERT

Al revisar una muestra de 50,000 registros se encontró que solo el **7.5%** de los textos combinados son únicos (3,760 de 50,000). Esto indica que el dataset es sintético: reutiliza un número relativamente pequeño de plantillas de texto (Job Title + Role + Qualifications + skills + Responsibilities + Job Description) y las replica con distintos países, empresas, salarios y modalidades.

Esto cambia la estrategia: en vez de generar un embedding por cada una de las 1,615,940 filas (carísimo y redundante, ya que muchas filas tendrían el **mismo vector exacto**), conviene:

1. **Deduplicar** por `texto_combinado` sobre **todo el dataset**, no solo una muestra.
2. Generar embeddings **solo de las plantillas únicas** (probablemente unos pocos miles).
3. Mantener un mapeo `template_id → todas las ofertas (Job Id, país, salario, modalidad, etc.) que comparten esa plantilla`.
4. Al buscar: encontrar la(s) plantilla(s) más similares con FAISS, y luego expandir a las ofertas reales que la comparten, aplicando ahí los filtros estructurados (país, modalidad, salario, experiencia).

Esto permite trabajar con el **dataset completo** (no una muestra) sin disparar el costo computacional, porque el costo de embeddings depende del número de plantillas únicas, no del número de filas.

### 2.1 Diagnóstico de duplicación en el dataset completo

Antes de decidir cuántas plantillas hay que embeber, se confirma la magnitud real sobre las 1,615,940 filas (no solo la muestra de 50,000 usada antes).

In [ ]:
n_total = len(df_clean)
n_unicos = df_clean["texto_combinado"].nunique()

print(f"Ofertas totales: {n_total:,}")
print(f"Textos combinados únicos: {n_unicos:,} ({n_unicos / n_total:.2%})")


Ofertas totales: 1,615,940
Textos combinados únicos: 3,760 (0.23%)


### 2.2 Construcción de la tabla de plantillas únicas

Se asigna un `template_id` a cada texto combinado distinto, y se construye una tabla `plantillas` con una fila por plantilla única — esa es la que se va a embeber. El `template_id` se agrega también a `df_clean` para poder expandir después los resultados de la búsqueda hacia todas las ofertas reales.

In [ ]:
df_clean["template_id"] = df_clean.groupby("texto_combinado", sort=False).ngroup()

plantillas = (
    df_clean
    .drop_duplicates(subset="template_id")
    .loc[:, ["template_id", "texto_combinado", "Job Title", "Role"]]
    .sort_values("template_id")
    .reset_index(drop=True)
)

print(f"Plantillas únicas a embeber: {len(plantillas):,}")
plantillas.head()


Plantillas únicas a embeber: 3,760


,template_id,texto_combinado,Job Title,Role
0,0,"Digital Marketing Specialist Digital Marketing Specialist Social Media Manager Social media platforms (e.g., Faceboo...",Digital Marketing Specialist,Social Media Manager
1,1,"Web Developer Web Developer Frontend Web Developer HTML, CSS, JavaScript Frontend frameworks (e.g., React, Angular) ...",Web Developer,Frontend Web Developer
2,2,Operations Manager Operations Manager Quality Control Manager Quality control processes and methodologies Statistica...,Operations Manager,Quality Control Manager
3,3,Network Engineer Network Engineer Wireless Network Engineer Wireless network design and architecture Wi-Fi standards...,Network Engineer,Wireless Network Engineer
4,4,Event Manager Event Manager Conference Manager Event planning Conference logistics Budget management Vendor coordina...,Event Manager,Conference Manager


### 2.3 Generación de embeddings (sobre las plantillas, no sobre las 1.6M filas)

Se usa `all-MiniLM-L6-v2`: modelo pequeño (384 dimensiones), rápido, buen punto de partida para *semantic search*. Si el desempeño no es suficiente en la fase de evaluación, se puede migrar a `all-mpnet-base-v2` (mejor calidad, más lento) en una segunda iteración — al ser pocas plantillas, cambiar de modelo es barato de volver a correr.

In [ ]:
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = "all-MiniLM-L6-v2"

modelo = SentenceTransformer(MODELO_EMBEDDINGS)
print(f"Modelo cargado: {MODELO_EMBEDDINGS}")
print(f"Dimensión del embedding: {modelo.get_sentence_embedding_dimension()}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado: all-MiniLM-L6-v2
Dimensión del embedding: 384


/tmp/ipykernel_4747/3411351968.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Dimensión del embedding: {modelo.get_sentence_embedding_dimension()}")


In [ ]:
inicio = time.time()

embeddings = modelo.encode(
    plantillas["texto_combinado"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # normaliza a norma 1 -> producto punto == similitud coseno
)

duracion = time.time() - inicio
print(f"Embeddings generados: {embeddings.shape}")
print(f"Tiempo total: {duracion:.1f} s  ({duracion / len(plantillas) * 1000:.2f} ms/plantilla)")


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Embeddings generados: (3760, 384)
Tiempo total: 8.2 s  (2.19 ms/plantilla)


In [ ]:
np.save(RUTA_PROCESSED / "embeddings_plantillas.npy", embeddings)
plantillas.to_parquet(RUTA_PROCESSED / "plantillas_meta.parquet", index=False)
df_clean[["Job Id", "template_id"]].to_parquet(RUTA_PROCESSED / "job_id_template_map.parquet", index=False)

print("Embeddings de plantillas y mapeo Job Id -> template_id guardados en Drive.")


Embeddings de plantillas y mapeo Job Id -> template_id guardados en Drive.


### 2.4 Búsqueda con FAISS (baseline)

Antes de explorar bases de datos vectoriales externas, se valida el pipeline con **FAISS** (tal como se planteó en el anteproyecto), usando un índice plano (`IndexFlatIP`) sobre las plantillas únicas.

In [ ]:
import faiss

dimension = embeddings.shape[1]
indice_faiss = faiss.IndexFlatIP(dimension)  # producto interno == coseno, porque los vectores están normalizados
indice_faiss.add(embeddings)

print(f"Plantillas indexadas en FAISS: {indice_faiss.ntotal:,}")


Plantillas indexadas en FAISS: 3,760


In [ ]:
def buscar_ofertas(consulta: str, k_plantillas: int = 5, max_resultados: int = 20, filtros: dict | None = None):
    """
    Busca ofertas semánticamente similares a `consulta`.

    - k_plantillas: cuántas plantillas de texto distintas considerar como relevantes.
    - max_resultados: máximo de ofertas individuales a devolver una vez expandidas las plantillas.
    - filtros: dict opcional de {columna: valor} sobre columnas de df_clean, p. ej. {"Country": "Colombia"}.
    """
    vector_consulta = modelo.encode([consulta], normalize_embeddings=True, convert_to_numpy=True)
    similitudes, indices = indice_faiss.search(vector_consulta, k_plantillas)

    template_ids_relevantes = plantillas.iloc[indices[0]]["template_id"].values
    similitud_por_template = dict(zip(template_ids_relevantes, similitudes[0]))

    resultados = df_clean[df_clean["template_id"].isin(template_ids_relevantes)].copy()
    resultados["similitud"] = resultados["template_id"].map(similitud_por_template)

    if filtros:
        for columna, valor in filtros.items():
            resultados = resultados[resultados[columna] == valor]

    resultados = resultados.sort_values("similitud", ascending=False)
    return resultados[["Job Title", "Role", "Country", "Work Type", "Salary Range", "similitud"]].head(max_resultados)

# Ejemplo sin filtros
buscar_ofertas("python developer with machine learning and NLP experience", k_plantillas=5)


,Job Title,Role,Country,Work Type,Salary Range,similitud
1579429,Data Scientist,Machine Learning Engineer,Philippines,Part-Time,$63K-$108K,0.528844
1564320,Data Scientist,Machine Learning Engineer,Netherlands,Contract,$57K-$89K,0.528844
8866,Data Scientist,Machine Learning Engineer,Burkina Faso,Temporary,$56K-$115K,0.528844
1578792,Data Scientist,Machine Learning Engineer,South Africa,Contract,$55K-$118K,0.528844
1568256,Data Scientist,Machine Learning Engineer,Malawi,Full-Time,$59K-$88K,0.528844
1564434,Data Scientist,Machine Learning Engineer,Portugal,Intern,$64K-$90K,0.528844
734900,Data Scientist,Machine Learning Engineer,Somalia,Full-Time,$59K-$93K,0.528844
738197,Data Scientist,Machine Learning Engineer,"Bahamas, The",Temporary,$61K-$108K,0.528844
739743,Data Scientist,Machine Learning Engineer,Moldova,Intern,$57K-$98K,0.528844
710950,Data Scientist,Machine Learning Engineer,British Virgin Islands,Full-Time,$60K-$106K,0.528844


In [ ]:
# Ejemplo aplicando un filtro estructurado, como se plantea en el Objetivo específico 4
buscar_ofertas(
    "python developer with machine learning and NLP experience",
    k_plantillas=5,
    filtros={"Work Type": "Full-Time"},
)


,Job Title,Role,Country,Work Type,Salary Range,similitud
1605706,Data Scientist,Machine Learning Engineer,Palau,Full-Time,$55K-$117K,0.528844
1555755,Data Scientist,Machine Learning Engineer,Andorra,Full-Time,$57K-$112K,0.528844
1459771,Data Scientist,Machine Learning Engineer,Guam,Full-Time,$55K-$97K,0.528844
1523747,Data Scientist,Machine Learning Engineer,Ghana,Full-Time,$57K-$87K,0.528844
1511162,Data Scientist,Machine Learning Engineer,Libya,Full-Time,$58K-$126K,0.528844
1529914,Data Scientist,Machine Learning Engineer,Kosovo,Full-Time,$63K-$112K,0.528844
1139789,Data Scientist,Machine Learning Engineer,Cambodia,Full-Time,$56K-$86K,0.528844
1548246,Data Scientist,Machine Learning Engineer,South Africa,Full-Time,$64K-$117K,0.528844
1548819,Data Scientist,Machine Learning Engineer,Sierra Leone,Full-Time,$61K-$81K,0.528844
405503,Data Scientist,Machine Learning Engineer,New Caledonia,Full-Time,$60K-$100K,0.528844
